List of the functions implemented in this notebook for the implementation of the "No-U-turn" algorithm for Hamiltonian Monte Carlo sampling:
1. L example (gasussian log-density) -> OK
2. Leapfrog -> Ok
3. pseudo_energy -> OK
4. FindReasonableEpsilon -> OK
5. BuildTree 

The functions are implemented in JAX in order to exploit its capabilitied to speed up the computations.

In [ ]:
# Usare numba con @njit

In [3]:
import math 
import numpy as np
import jax.random as jrnd
import jax.numpy as jnp
from jax import grad, jit
from numpy.typing import NDArray
from typing import Callable

In [6]:
#https://docs.jax.dev/en/latest/
# jax example for AD
import jax.numpy as jnp
from jax import grad

def f(x):
    return jnp.sin(x)

df = grad(f)
print(df(0.))

1.0


In [ ]:
# Gaussian log-density

@jit
def L(theta):
    return -0.5 * jnp.dot(theta, theta)  


In [ ]:
'''
1. definire L fuori
2. definire grad_L = grad(L) fuori
3. passare solo i valori, non la funzione
'''

grad_L = grad(L)

@jit
def Leapfrog(theta: NDArray[jnp.float64],
             r: NDArray[jnp.float64],
             epsilon: float) 
                -> tuple[NDArray[jnp.float64], NDArray[jnp.float64]]:

    # half-step momentum update
    r_tilde = r + 0.5 * epsilon * grad_L(theta)

    # full-step position update
    theta_tilde = theta + epsilon * r_tilde

    # second half-step momentum update
    r_tilde = r_tilde + 0.5 * epsilon * grad_L(theta_tilde)

    return theta_tilde, r_tilde


In [ ]:
# This calculate a value proportional to log(p(r,theta)) that is the hamiltonian value at r,theta;
# Working in log-space provide more stability and is useful in order to avoid overflows

@jit
def log_p(theta: NDArray[float], r: NDArray[float]) -> float:
    E = L(theta) - (0.5*jnp.dot(r,r))
    return E

In [ ]:
# Alghorithm 4 of the article

def FindReasonableEpsilon(theta: NDArray[float], key=jnp.array([0, 0])) -> float:

    # sample momentum r ~ N(0, I)
    key, subkey = jrnd.split(key)
    r = jrnd.normal(subkey, shape=theta.shape)

    # initial epsilon
    epsilon = 1.0

    # compute initial log joint density
    E0 = log_p(theta, r)

    # one leapfrog step
    theta_prime, r_prime = Leapfrog(theta, r, epsilon)
    E1 = log_p(theta_prime, r_prime)

    # log acceptance probability
    log_accept = E1 - E0

    # direction: +1 (increase) or -1 (decrease)
    a = jnp.where(log_accept > jnp.log(0.5), 1.0, -1.0)

    # -------------------------------
    # cond_fun: while loop consition
    # -------------------------------
    def cond_fun(state):
        eps, log_acc = state
        return (a * log_acc) > (-a * jnp.log(2.0))

    # --------------------
    # body_fun: loop body
    # --------------------
    def body_fun(state):
        eps, _ = state
        eps = eps * (2.0 ** a)
        theta_p, r_p = Leapfrog(theta, r, eps)
        E1 = log_p(theta_p, r_p)
        log_acc = E1 - E0
        return (eps, log_acc)

    # ciclo while JAX
    epsilon, _ = jax.lax.while_loop(cond_fun, body_fun, (epsilon, log_accept))

    return epsilon


In [ ]:
# input: [theta, r, u, v, j, epsilon, theta_0, r_0]
# theta,r -> coordinates
# u -> slice variable
# v -> {+1,-1} direction
# j -> number of previous doublings

def BuildTree(root: NDArray[float]) -> NDArray[float]:

    # retrieve the variables from the input
    theta, r, u, v, j, epsilon, theta_0, r_0 = root
    
    if j == 0:
        # Base case -> take one Leapfrog step in the v direction
        new_r, new_theta = Leapfrog(theta, r, v*epsilon)
        n_prime = (jnp.log(u) <= log_p(new_theta, new_r)
        s_prime = #...
        return #...

    else:
        


        
    return 